# Fine-tuning Dataset Preparation – PubMedQA

Este notebook prepara o dataset **PubMedQA (PQA-L)** para fine-tuning de um modelo LLM no Azure AI Foundry.

## Objetivo

Transformar o dataset original em um formato adequado para treinamento de modelos conversacionais.

## Etapas

1. Carregar o dataset original
2. Analisar a estrutura dos dados
3. Realizar preprocessing
4. Criar dataset estruturado
5. Converter para formato de fine-tuning (JSONL chat format)

## Dataset utilizado

PubMedQA – PQA-L (labeled subset)

Este dataset contém:
- Perguntas médicas
- Contexto científico (abstracts)
- Respostas baseadas em evidência

O dataset PubMedQA já é composto por abstracts científicos e não contém dados pessoais de pacientes, portanto nenhuma etapa adicional de anonimização foi necessária.

### Bibliotecas

In [1]:
import json
import pandas as pd
from pathlib import Path

### Definimos os caminhos para:

- dataset original
- dataset processado
- dataset final no formato necessário para fine-tuning

In [8]:
BASE_PATH = Path("B:/Pós/hospital-ai-diagnosis/dados/fine_tuning")

RAW_PATH = BASE_PATH / "Raw" / "ori_pqal.json"
PROCESSED_PATH = BASE_PATH / "Processados" / "ori_pqal_processed.json"
AZURE_PATH = BASE_PATH / "llm_ready" / "pubmedqa_finetune.jsonl"

print(RAW_PATH)

B:\Pós\hospital-ai-diagnosis\dados\fine_tuning\Raw\ori_pqal.json


### Carregamento do dataset original

O PubMedQA possui uma estrutura onde:

- cada chave representa um PubMed ID
- cada valor contém:
  - QUESTION
  - CONTEXTS
  - LONG_ANSWER
  - final_decision

In [9]:
with open(RAW_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total registros no dataset: {len(data)}")

Total registros no dataset: 1000


Esta etapa permite visualizar a estrutura completa de um registro do dataset.

Campos importantes:

- QUESTION → pergunta clínica
- CONTEXTS → abstracts científicos
- LONG_ANSWER → resposta detalhada
- final_decision → resposta curta (yes/no/maybe)

In [10]:
first_key = list(data.keys())[0]

data[first_key]

{'QUESTION': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'CONTEXTS': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
  'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD)

### O preprocessing realiza:

1. Extração dos campos relevantes
2. Junção dos abstracts em um único texto
3. Remoção de registros incompletos
4. Criação de uma estrutura padronizada

In [11]:
def preprocess_pubmedqa(dataset):

    processed = []

    for pmid, entry in dataset.items():

        question = entry.get("QUESTION", "").strip()
        contexts = entry.get("CONTEXTS", [])
        answer = entry.get("LONG_ANSWER", "").strip()
        decision = entry.get("final_decision", "").strip()

        if not question or not answer:
            continue

        context_text = "\n\n".join(contexts)

        processed.append({
            "pmid": pmid,
            "question": question,
            "context": context_text,
            "answer": answer,
            "decision": decision
        })

    return processed

### Execução do pre-processamento

In [12]:
processed_data = preprocess_pubmedqa(data)

print("Total exemplos processados:", len(processed_data))

Total exemplos processados: 1000


### Convertendo os dados para DataFrame podemos:

- visualizar amostras
- verificar qualidade
- validar estrutura

In [13]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(PROCESSED_PATH, "w", encoding="utf-8") as f:
    json.dump(processed_data, f, indent=2, ensure_ascii=False)

print("Dataset processado salvo.")

Dataset processado salvo.


### Converter para formato de fine-tuning (Azure)

Azure exige JSONL com formato de chat.

In [15]:
def build_azure_format(dataset):

    azure_data = []

    for entry in dataset:

        messages = [
            {
                "role": "system",
                "content": "You are a medical research assistant that answers questions based on scientific evidence."
            },
            {
                "role": "user",
                "content": f"Context:\n{entry['context']}\n\nQuestion: {entry['question']}"
            },
            {
                "role": "assistant",
                "content": entry["answer"]
            }
        ]

        azure_data.append({"messages": messages})

    return azure_data

### Geração do dataset final

In [17]:
azure_dataset = build_azure_format(processed_data)

len(azure_dataset)


AZURE_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(AZURE_PATH, "w", encoding="utf-8") as f:

    for row in azure_dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Dataset pronto para fine-tuning salvo.")

Dataset pronto para fine-tuning salvo.


### Validação final

In [18]:
azure_dataset[0]

{'messages': [{'role': 'system',
   'content': 'You are a medical research assistant that answers questions based on scientific evidence.'},
  {'role': 'user',
   'content': 'Context:\nProgrammed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.\n\nThe following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that wil

## Resultado

O dataset foi convertido para formato compatível com fine-tuning de LLMs.

Etapas realizadas:

- análise do dataset PubMedQA
- preprocessing e curadoria
- estruturação dos dados
- conversão para formato de treinamento

Este dataset poderá ser utilizado para treinar uma LLM no Azure AI Foundry.